## We test ***LLM's response by passing wrong inputs*** at *3 separate levels*:

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

## 1.Install Required Libraries

In [1]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)

Enter your OpenAI API key:  ········


OpenAI client configured successfully.
Model: gpt-4.1-mini


## 2.1 Pythonn Application functions

In [2]:
def employee_lookup(employee_id):
    employees = {101: {"name": "Madhavan", "department": "AI Research Team"},
                 102: {"name": "Rajesh", "department": "Operations"},
                 103: {"name": "Jagadeeshwari", "department": "AI Engineering Team"}}
    return employees.get(employee_id,{"error": "Employee Not Found"})

def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

def calculate(expression):
    allowed = set("0123456789+-*/(). %")

    if not expression or any(char not in allowed for char in expression):
        return {"error": "Only basic arithmetic expressions are allowed."}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception:
        return {"error": "Invalid arithmetic expression."}

def send_email(to, subject, body):
    return {"status": "simulated","message": f"Email prepared for {to} with subject '{subject}'." }

def book_meeting(title, attendee, time):
    return {"status": "simulated","message": f"Meeting '{title}' prepared with {attendee} at {time}."}

## 2.2 LLM Input - expects Tools

In [3]:
TOOLS = [
    {
        "type": "function",
        "name": "employee_lookup",
        "description": "Look up an employee's name and department using an employee ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "employee_id": {"type": "integer"}
            },
            "required": ["employee_id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate remaining leave from total and taken leaves.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {"type": "integer"},
                "leaves_taken": {"type": "integer"}
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform a basic arithmetic calculation.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "Prepare a simulated email.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "book_meeting",
        "description": "Prepare a simulated meeting booking.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "attendee": {"type": "string"},
                "time": {"type": "string"}
            },
            "required": ["title", "attendee", "time"],
            "additionalProperties": False
        }
    }
]

print("Available tools:", [tool["name"] for tool in TOOLS])

Available tools: ['employee_lookup', 'calculate_leave_balance', 'calculate', 'send_email', 'book_meeting']


## 3.Tool Registry Preparation

Why Tool Registry? A registry connects the model's selected tool name with the corresponding python function.

In [4]:
TOOL_REGISTRY = {"employee_lookup": employee_lookup,
                 "calculate_leave_balance": calculate_leave_balance,
                 "calculate": calculate,
                 "send_email": send_email,
                 "book_meeting": book_meeting}

## 4.The model can select the appropriate tool based on the user's request.

## 4.1. Without Tools

In [10]:
question = "Who is employee 102?"

response=client.responses.create(model=GENERATION_MODEL,input=question)

print(response.output_text) #It'll give a generic information as output

I don’t see any employee information in the current context. Could you please provide more details or share the relevant document or image where employee 102 is mentioned?


## 4.2 With Tool

In [12]:
question="Who is employee 102?"

response=client.responses.create(model=GENERATION_MODEL,input=question,tools=TOOLS)

print(response.output)

[ResponseFunctionToolCall(arguments='{"employee_id":102}', call_id='call_s2nTpKV8zkDyi6PGoDA81h2r', name='employee_lookup', type='function_call', id='fc_086a66e341d27909006ab02108f35887d1aae64eb97c08c225', async_=None, caller=None, namespace=None, status='completed')]


In [15]:
for item in response.output:
    if item.type == "function_call":
        print("Selected Tool:",item.name)
        print("Arguments:",item.arguments)

Selected Tool: employee_lookup
Arguments: {"employee_id":102}


## Main Parts of this notebook.

We test ***LLM's response by passing wrong inputs*** at *3 separate levels*:

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

### 1.User Prompt Level

The user's Prompt itself is incomplete or invalid.

In [16]:
question = "Find employee AB and tell me their name and department."

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)

print("-"*30)
print(response.output)
print("-"*30)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)
    else:
        print("No tool call was produced.")
        print("Final output:", response.output_text)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":1}', call_id='call_XIuY2ecE9V40FeoA3g4wb7RN', name='employee_lookup', type='function_call', id='fc_05863b2029ef7633006ab02309391087d18d97172f0f846ee0', async_=None, caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":1}


#### **Add this step: Define the "Client Response Extractor"**

In [17]:
def client_response_extractor(question):
  response = client.responses.create(model=GENERATION_MODEL,
                                    input=question,
                                    tools=TOOLS)
  print("-"*30)
  print(response.output)
  print("-"*30)

  for item in response.output:
      if item.type == "function_call":
          print("Selected tool:", item.name)
          print("Arguments:", item.arguments)
      else:
          print("No tool call was produced.")
          print("Final output:", response.output_text)

In [18]:
#unknown employee id

question = "Find employee 999 and tell me their name and department."
client_response_extractor(question)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":999}', call_id='call_SQm0GzrNliB59LHzyV7jIJ4m', name='employee_lookup', type='function_call', id='fc_04e9cca7ea737344006ab02348f8ac87d181eaac051f98a466', async_=None, caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":999}


In [19]:
#Missing leave values

question = "Calculate the leave balance, but I will not give you total leaves or leaves taken."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_0f4fc3f11399317c006ab023648ccc87d1afea860ced0890f7', content=[ResponseOutputText(annotations=[], text='To calculate the leave balance, I need the total number of leaves and the number of leaves taken. Could you please provide those details?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: To calculate the leave balance, I need the total number of leaves and the number of leaves taken. Could you please provide those details?


In [20]:
#Division by zero

question = "Calculate 10 divided by zero."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_09df5b75df69b7af006ab0237a69cc87d1a1cf3196271a8ebf', content=[ResponseOutputText(annotations=[], text='Division by zero is undefined in mathematics. It is not possible to divide any number by zero. If you have another calculation or question, feel free to ask!', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: Division by zero is undefined in mathematics. It is not possible to divide any number by zero. If you have another calculation or question, feel free to ask!


In [21]:
#Unsupported calculation

question = "Calculate DROP TABLE employees."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_0c5ce4832bea478f006ab024459b0087d1ba83ac5c6df6667a', content=[ResponseOutputText(annotations=[], text='It looks like you are asking for a calculation related to the phrase "DROP TABLE employees." This phrase is actually a SQL command used to delete the entire "employees" table from a database. It is not a mathematical expression that can be calculated.\n\nCould you please clarify if you want to perform a calculation or if you need help with a SQL command? If you want a calculation or assistance with SQL syntax, please provide more details.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: It looks like you are asking for a calculation related to the phrase "DROP TABLE employees." This phrase is actually a SQL command used to delete the entire "employees" table from a database. It is not a mathematical

In [23]:
#Invalid enmail recipient

question = "Prepare an email to not-an-email saying hello."
client_response_extractor(question)

------------------------------
[ResponseFunctionToolCall(arguments='{"to":"not-an-email","subject":"Hello","body":"Hello"}', call_id='call_R5PdIIJZPtBHBa1aJ8lrK2Lk', name='send_email', type='function_call', id='fc_0d3a430773240ce8006ab02460d8fc87d18979d5a061fed394', async_=None, caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: send_email
Arguments: {"to":"not-an-email","subject":"Hello","body":"Hello"}


In [24]:
#Missing meeting title

question = "Book a meeting with John tomorrow, but I have not provided a meeting title."
client_response_extractor(question)

------------------------------
[ResponseOutputMessage(id='msg_0f8b8cb521aea843006ab0248a11b487d19ff3466e7b112fef', content=[ResponseOutputText(annotations=[], text='Could you please provide the preferred time for the meeting tomorrow and a title or topic for the meeting? This will help me to book the meeting accurately.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: Could you please provide the preferred time for the meeting tomorrow and a title or topic for the meeting? This will help me to book the meeting accurately.


### Level 2 and 3 : Tool Argument Level & Tool execution level validation

### Reminder:
1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

We will intentionally pass incorrect arguments to each tool.

In [25]:
#Defining the tool safety settings or tool safety wrappper

import inspect
import json

def execute_tool_safely(tool_name, arguments):
    if tool_name not in TOOL_REGISTRY:
        return {"status": "error","error_type": "UnknownTool","error_message": f"Unknown tool requested: {tool_name}"}

    function = TOOL_REGISTRY[tool_name]

    try:
        # Python signature validation: catches missing and unexpected arguments.
        bound = inspect.signature(function).bind(**arguments)

        # Explicit type validation for demonstration.
        if tool_name == "employee_lookup":
            if not isinstance(arguments.get("employee_id"),int):
                raise TypeError("employee_id must be an integer.")

        elif tool_name == "calculate_leave_balance":
            for name in ("total_leaves", "leaves_taken"):
                value = arguments.get(name)
                if not isinstance(value, int):
                    raise TypeError(f"{name} must be an integer.")

        elif tool_name == "calculate":
            if not isinstance(arguments.get("expression"), str):
                raise TypeError("expression must be a string.")

        elif tool_name == "send_email":
            for name in ("to", "subject", "body"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if "@" not in arguments["to"]:
                raise ValueError("Invalid email address.")

        elif tool_name == "book_meeting":
            for name in ("title", "attendee", "time"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if not arguments["title"].strip():
                raise ValueError("Meeting title cannot be empty.")

        result = function(**bound.arguments)
        return {"status": "success","result": result}

    except TypeError as e:
        return {"status": "error","error_type": "TypeError","error_message": str(e)}

    except ValueError as e:
        return {"status": "error","error_type": "ValueError","error_message": str(e)}

    except Exception as e:
        return {"status": "error","error_type": type(e).__name__,"error_message": str(e)}

print("Safe tool execution wrapper ready.")

Safe tool execution wrapper ready.


In [27]:
#Negative test 1: employee_lookup - Wrong type

result = execute_tool_safely("employee_lookup", {"employee_id": "ABC"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'employee_id must be an integer.'}


In [28]:
#Negative test 2: employee_lookup - missing parameter

result = execute_tool_safely("employee_lookup", {})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'employee_id'"}


In [29]:
# negative 3:employee_lookup - Unknown employee

result = execute_tool_safely("employee_lookup", {"employee_id": 999})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Employee Not Found'}}


In [30]:
#negative 4: calculate_leave_balance - wrong type

result = execute_tool_safely("calculate_leave_balance", {"total_leaves": "24", "leaves_taken": 7})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'total_leaves must be an integer.'}


In [31]:
result = execute_tool_safely("calculate_leave_balance", 24)

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'inspect.Signature.bind() argument after ** must be a mapping, not int'}


In [32]:
#negative test 5: calculate_leave_balance - missing parameter

result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'leaves_taken'"}


In [33]:
#Negative test 6: calculate_leave_balance - unexpected parameter

result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": 7, "extra": 1})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "got an unexpected keyword argument 'extra'"}


In [34]:
#Negative Test 6: calculate_leave_balance — unexpected parameter

result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": 7, "extra": 1})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "got an unexpected keyword argument 'extra'"}


In [35]:
#Negative Test 7: calculate_leave_balance — negative leaves taken

result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": -5})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'remaining_leaves': 29}}


In [36]:
#Negative Test 8: calculate — division by zero
result = execute_tool_safely("calculate", {"expression": "10 / 0"})

print("Tool Result:")
print(result)


Tool Result:
{'status': 'success', 'result': {'error': 'Invalid arithmetic expression.'}}


In [37]:
#Negative Test 9: calculate — unsupported expression

result = execute_tool_safely("calculate", {"expression": "DROP TABLE employees"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Only basic arithmetic expressions are allowed.'}}


In [38]:
#Negative Test 10: send_email — wrong subject type

result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": 123, "body": "Hello"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'subject must be a string.'}


In [39]:
#Negative Test 11: book_meeting — missing time

result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": "John"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'time'"}


# Running the agent

1. Receive User Input
2. Store Conversation
3. Start Agent Loop
4. Send Request to LLM
5. Check for Tool Call
6. Save LLM Response
7. Execute Selected Tool
8. Get Tool Result
9. Send Tool Result Back to LLM
10. Repeat Until Final Answer or Maximum Steps Reached

In [45]:
import json

def run_agent_with_error_handling(user_input, max_steps=5):
    input_items = [{"role": "user", "content": user_input}]

    for step in range(1, max_steps + 1):
        response = client.responses.create(model=GENERATION_MODEL,
                                           input=input_items,
                                           tools=TOOLS )

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            return response.output_text

        input_items.extend([item.model_dump(by_alias=True) for item in response.output]) #--> part 1 work is done

        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.arguments)
                tool_result = execute_tool_safely(tool_call.name, arguments)
            except json.JSONDecodeError as e:
                tool_result = {"status": "error","error_type": "InvalidJSON","error_message": f"Could not parse tool arguments: {e}"}

            print(f"Step {step}")
            print("Tool:", tool_call.name)
            print("Arguments:", arguments)
            print("Result:", tool_result)

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(tool_result)
            })

    return "Maximum agent steps reached."

In [46]:
# Single step task
run_agent_with_error_handling("Find employee 102 and tell me their name and department.")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 102}
Result: {'status': 'success', 'result': {'name': 'Rajesh', 'department': 'Operations'}}


'Employee 102 is named Rajesh and he works in the Operations department.'

In [47]:
#Multi step task
run_agent_with_error_handling("""Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and
                               department and asks HR to confirm the employee's leave balance.""")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 103}
Result: {'status': 'success', 'result': {'name': 'Jagadeeshwari', 'department': 'AI Engineering Team'}}
Step 2
Tool: send_email
Arguments: {'to': 'hr@company.com', 'subject': 'Leave Balance Confirmation Request for Employee Jagadeeshwari', 'body': 'Dear HR Team,\n\nI hope this message finds you well. I am writing to request confirmation of the leave balance for one of our employees, Jagadeeshwari, who is part of the AI Engineering Team.\n\nCould you please provide me with the details of her current leave balance at your earliest convenience?\n\nThank you for your assistance.\n\nBest regards,\n[Your Name]'}
Result: {'status': 'success', 'result': {'status': 'simulated', 'message': "Email prepared for hr@company.com with subject 'Leave Balance Confirmation Request for Employee Jagadeeshwari'."}}


'The employee with ID 103 is Jagadeeshwari from the AI Engineering Team. I have prepared a simulated email to HR requesting confirmation of her leave balance. If you want me to send or modify the email, please let me know.'